# Learned Stage-Wise Curvature Tuning on Arabic Characters

This notebook evaluates Learned Stage-Wise Curvature Tuning (SW-CT) on the Arabic Handwritten Characters dataset using an ImageNet-pretrained ResNet-50.

Arabic Characters is a handwritten character-recognition task that differs substantially from the natural-image domain used for ImageNet pretraining. This provides a strong domain-shift experiment for evaluating whether stage-wise curvature adaptation can transfer to a visually different classification problem.

The experiments compare four conditions:

1. **Baseline** — the original pretrained ResNet with ReLU activations and a trained downstream classifier.
2. **Single-Parameter Curvature Tuning (S-CT)** — one global beta selected through validation-based exhaustive search.
3. **Learned SW-CT with fixed initialization** — four trainable stage-wise beta parameters initialized at 0.78.
4. **Learned SW-CT with random initialization** — four trainable stage-wise beta parameters independently initialized between 0.70 and 0.99.

Each condition is evaluated using seeds 42, 43, and 44.

Experiments are conducted for both 20 and 30 epochs to determine whether additional optimization improves classification performance or changes the learned stage-wise beta values.

Wall-clock execution time is also recorded to compare computational cost across Baseline, S-CT, and Learned SW-CT.

In [1]:
!nvidia-smi

Sun Oct  4 23:59:59 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
%cd /content

!git clone -b main \
    https://github.com/NiftySnek173/research-curvature-tuning.git

%cd /content/research-curvature-tuning/src/curvature-tuning

!git branch --show-current
!git log -1 --oneline

/content
Cloning into 'curvature-tuning-research'...
remote: Enumerating objects: 592, done.
remote: Counting objects: 100% (264/264), done.
remote: Compressing objects: 100% (128/128), done.
remote: Total 592 (delta 166), reused 198 (delta 115), pack-reused 328 (from 1)
Receiving objects: 100% (592/592), 17.47 MiB | 7.75 MiB/s, done.
Resolving deltas: 100% (297/297), done.
/content/curvature-tuning-research/src/curvature-tuning
stage-wise-ct
6ec796c (HEAD -> stage-wise-ct, origin/stage-wise-ct, origin/HEAD) updated the notebook with outputs


## Environment Setup

The repository dependencies are installed before running the experiments.

After installation, the Colab runtime should be restarted so that the required package versions are loaded correctly.

In [3]:
%pip install -q -r requirements.txt
%pip install --force-reinstall --no-cache-dir "numpy==1.26.3"

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.7/15.7 MB 47.2 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 1.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.2/133.2 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 47.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.4/78.4 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 32.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [1]:
%cd /content/research-curvature-tuning/src/curvature-tuning

import os
import torch

os.environ["WANDB_MODE"] = "disabled"

print("Working directory:", os.getcwd())
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

/content/curvature-tuning-research/src/curvature-tuning
Working directory: /content/curvature-tuning-research/src/curvature-tuning
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


## Arabic Digits Dataset

Before running the experiments, the Arabic Digits data loaders are checked to confirm that the downstream dataset can be loaded correctly.

The same dataset split and random seed protocol are used for Baseline, S-CT, and Learned SW-CT so that the methods are compared under the same conditions.

In [2]:
from utils.data import get_data_loaders

train_loader, test_loader, val_loader = get_data_loaders(
    "imagenet_to_arabic-characters",
    train_batch_size=32,
    test_batch_size=128,
    seed=42,
)

print("Train samples:", len(train_loader.dataset))
print("Validation samples:", len(val_loader.dataset))
print("Test samples:", len(test_loader.dataset))

images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

README.md:   0%|          | 0.00/2.65k [00:00<?, ?B/s]

arabic-characters.py:   0%|          | 0.00/5.08k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Processing train set: 100%|██████████| 13441/13441 [00:03<00:00, 3815.02it/s]


Generating test split: 0 examples [00:00, ? examples/s]

Processing test set: 100%|██████████| 3361/3361 [00:00<00:00, 4206.57it/s]


Train samples: 10752
Validation samples: 2688
Test samples: 3360
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])
Labels: tensor([ 9, 16, 17, 14,  6, 24, 15, 10, 26, 15])


# ResNet-50 Experiments

ResNet-50 is evaluated first because it was used during the initial development of Learned SW-CT.

The experiment compares Baseline, S-CT, fixed-initialization SW-CT, and random-initialization SW-CT.

All methods use the same ImageNet-pretrained backbone, Arabic Digits dataset split, experimental seeds, and downstream training protocol.

## ResNet-50 — 20-Epoch Evaluation

The first evaluation uses 20 training epochs.

This provides the primary reference experiment and maintains compatibility with the earlier Learned SW-CT work.

### Methods

**Baseline**

The pretrained ResNet-50 retains its original ReLU activations. The backbone is frozen and only the downstream classifier is trained.

**S-CT**

The original Single-Parameter Curvature Tuning method searches over a single beta shared throughout the network. Candidate beta values from 0.70 to 1.00 are evaluated using validation performance, and the best candidate is evaluated on the test set.

The recorded S-CT execution time includes the complete beta search rather than only the final selected beta.

**Learned SW-CT — Fixed Initialization**

Four trainable beta parameters are assigned to the four ResNet stages. All four begin at beta = 0.78.

This preserves the initialization used in the original Learned SW-CT experiments.

**Learned SW-CT — Random Initialization**

Four trainable stage-wise beta parameters are initialized independently between 0.70 and 0.99.

The random initialization is generated deterministically from the experiment seed. This allows the experiment to test whether Learned SW-CT depends on the fixed 0.78 starting value while remaining reproducible.

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-03 20:50:16.549 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_arabic-characters_resnet50_seed42_epochs20.log
2026-10-03 20:50:16.551 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 20:50:38.591 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 20:50:38.655 | INFO     | __main__:main:116 - Number of trainable parameters: 57372
2026-10-03 20:50:38.655 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 20:51:20.290 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.350443, Accuracy: 9.38%
2026-10-03 20:51:20.651 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 3.273959, Accuracy: 7.33%
2026-10-03 20:51:21.066 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 3.074416, Accuracy: 16.45%
2026-10-03 20:51:21.395 | INFO     | train:train_epoch:47 - Epoch 1, Step 300, Loss: 2.862928, Accuracy: 22.97%
2026-10-03 20:51:21.761 | INFO     | train:test_epoch:90 - Epoch 1, Val Los

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-03 21:53:16.557 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_arabic-characters_resnet50_seed43_epochs20.log
2026-10-03 21:53:16.562 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 21:53:38.801 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 21:53:38.880 | INFO     | __main__:main:116 - Number of trainable parameters: 57372
2026-10-03 21:53:38.881 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 21:54:23.683 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.356981, Accuracy: 3.12%
2026-10-03 21:54:24.047 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 3.269083, Accuracy: 9.10%
2026-10-03 21:54:24.412 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 3.077229, Accuracy: 17.60%
2026-10-03 21:54:24.764 | INFO     | train:train_epoch:47 - Epoch 1, Step 300, Loss: 2.867395, Accuracy: 23.94%
2026-10-03 21:54:25.162 | INFO     | train:test_epoch:90 - Epoch 1, Val Los

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 20

2026-10-03 23:00:40.699 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_arabic-characters_resnet50_seed44_epochs20.log
2026-10-03 23:00:40.704 | INFO     | __main__:main:78 - Running on cuda
2026-10-03 23:01:04.781 | INFO     | __main__:main:105 - Testing baseline...
2026-10-03 23:01:04.853 | INFO     | __main__:main:116 - Number of trainable parameters: 57372
2026-10-03 23:01:04.854 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-03 23:01:51.105 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.327988, Accuracy: 6.25%
2026-10-03 23:01:51.536 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 3.267558, Accuracy: 9.44%
2026-10-03 23:01:51.901 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 3.083898, Accuracy: 16.14%
2026-10-03 23:01:52.253 | INFO     | train:train_epoch:47 - Epoch 1, Step 300, Loss: 2.883959, Accuracy: 21.98%
2026-10-03 23:01:52.677 | INFO     | train:test_epoch:90 - Epoch 1, Val Los

### Learned SW-CT with Fixed Beta Initialization

The original Learned SW-CT configuration is first reproduced using the fixed initialization:

\[[\beta_1,\beta_2,\beta_3,\beta_4]=[0.78,0.78,0.78,0.78].\]

The initial value is only a starting point. During training, the four beta parameters are optimized independently together with the downstream classifier.

These runs provide the reference against which random initialization will be compared.

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-04 00:06:52.527 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet50_seed42_epochs20_initfixed0.78_betalr0.01.log
2026-10-04 00:06:52.596 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 00:07:13.493 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-04 00:07:13.494 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 00:07:13.566 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-04 00:07:13.567 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 00:07:13.568 | INFO     | __main__:main:439 - Total trainable parameters: 57376
2026-10-04 00:07:14.986 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.392796, Accuracy: 3.12%
2026-10-04 00:08:11.470 | INFO     | train:tra

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-04 01:18:00.900 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet50_seed43_epochs20_initfixed0.78_betalr0.01.log
2026-10-04 01:18:00.960 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 01:18:19.429 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-04 01:18:19.430 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 01:18:19.488 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-04 01:18:19.488 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 01:18:19.489 | INFO     | __main__:main:439 - Total trainable parameters: 57376
2026-10-04 01:18:20.836 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.292988, Accuracy: 9.38%
2026-10-04 01:19:17.702 | INFO     | train:tra

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 20

2026-10-04 02:29:00.319 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet50_seed44_epochs20_initfixed0.78_betalr0.01.log
2026-10-04 02:29:00.376 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 02:29:18.459 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-04 02:29:18.459 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 02:29:18.552 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-04 02:29:18.554 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 02:29:18.554 | INFO     | __main__:main:439 - Total trainable parameters: 57376
2026-10-04 02:29:19.858 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.353614, Accuracy: 3.12%
2026-10-04 02:30:16.671 | INFO     | train:tra

### Learned SW-CT with Random Beta Initialization

The fixed beta = 0.78 initialization may raise the question of whether Learned SW-CT performs well only because its starting point was informed by the earlier S-CT experiments.

To test this dependency, each of the four stage-wise beta parameters is now initialized independently from a uniform distribution between 0.70 and 0.99.

The initialization is reproducible: the random generator is seeded using the experiment seed. Therefore, each seed always receives the same initial beta vector.

If random initialization produces similar performance to the fixed 0.78 experiment, this provides evidence that the Learned SW-CT result is not dependent on a single hand-selected initialization.

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-04 03:40:05.547 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet50_seed42_epochs20_initrandom_betalr0.01.log
2026-10-04 03:40:05.673 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 03:40:25.630 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-04 03:40:25.630 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 03:40:25.699 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-04 03:40:25.700 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 03:40:25.700 | INFO     | __main__:main:439 - Total trainable parameters: 57376
2026-10-04 03:40:27.111 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.396827, Accuracy: 3.12%
2026-10-04 03:41:25.184 | INFO    

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-04 04:51:18.416 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet50_seed43_epochs20_initrandom_betalr0.01.log
2026-10-04 04:51:18.507 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 04:51:37.612 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-04 04:51:37.613 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 04:51:37.678 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-04 04:51:37.679 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 04:51:37.679 | INFO     | __main__:main:439 - Total trainable parameters: 57376
2026-10-04 04:51:39.082 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.315885, Accuracy: 9.38%
2026-10-04 04:52:37.113 | INFO  

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 20

2026-10-04 06:02:18.857 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet50_seed44_epochs20_initrandom_betalr0.01.log
2026-10-04 06:02:18.943 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 06:02:36.864 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-04 06:02:36.865 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 06:02:36.926 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-04 06:02:36.927 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 06:02:36.927 | INFO     | __main__:main:439 - Total trainable parameters: 57376
2026-10-04 06:02:38.221 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.344718, Accuracy: 0.00%
2026-10-04 06:03:35.696 | INFO  

# ResNet-50 — 30-Epoch Evaluation

The same experiments are now repeated for 30 epochs.

The purpose of increasing the training budget is to determine whether the 20-epoch results are limited by insufficient optimization.

If a method improves substantially with 30 epochs, this suggests that additional optimization is useful. If performance remains similar, it indicates that the method has largely stabilized within the original 20-epoch training budget.

All other experimental settings remain unchanged.

For random-initialization SW-CT, the same seed produces exactly the same initial four-beta vector in both the 20-epoch and 30-epoch experiments. Therefore, initialization is held constant while the training duration changes.

Execution time is again recorded to measure the computational cost of extending training from 20 to 30 epochs.

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 42 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-04 07:13:20.446 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_arabic-characters_resnet50_seed42_epochs30.log
2026-10-04 07:13:20.450 | INFO     | __main__:main:78 - Running on cuda
2026-10-04 07:13:38.689 | INFO     | __main__:main:105 - Testing baseline...
2026-10-04 07:13:38.751 | INFO     | __main__:main:116 - Number of trainable parameters: 57372
2026-10-04 07:13:38.751 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-04 07:14:25.356 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.350443, Accuracy: 9.38%
2026-10-04 07:14:25.733 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 3.273959, Accuracy: 7.33%
2026-10-04 07:14:26.099 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 3.074416, Accuracy: 16.45%
2026-10-04 07:14:26.468 | INFO     | train:train_epoch:47 - Epoch 1, Step 300, Loss: 2.862928, Accuracy: 22.97%
2026-10-04 07:14:26.836 | INFO     | train:test_epoch:90 - Epoch 1, Val Los

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 43 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-04 11:30:12.996 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_arabic-characters_resnet50_seed43_epochs30.log
2026-10-04 11:30:12.999 | INFO     | __main__:main:78 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 226MB/s]
2026-10-04 11:30:29.124 | INFO     | __main__:main:105 - Testing baseline...
2026-10-04 11:30:29.173 | INFO     | __main__:main:116 - Number of trainable parameters: 57372
2026-10-04 11:30:29.173 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-04 11:31:10.003 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.356981, Accuracy: 3.12%
2026-10-04 11:31:10.287 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 3.269083, Accuracy: 9.10%
2026-10-04 11:31:10.565 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 44 \
    --linear_probe_train_bs 32 \
    --linear_probe_test_bs 128 \
    --epochs 30

2026-10-04 12:43:18.452 | INFO     | __main__:main:74 - Log file: ./logs/generalization_ct_imagenet_to_arabic-characters_resnet50_seed44_epochs30.log
2026-10-04 12:43:18.456 | INFO     | __main__:main:78 - Running on cuda
2026-10-04 12:43:43.558 | INFO     | __main__:main:105 - Testing baseline...
2026-10-04 12:43:43.640 | INFO     | __main__:main:116 - Number of trainable parameters: 57372
2026-10-04 12:43:43.641 | INFO     | __main__:main:117 - Starting transfer learning...
2026-10-04 12:44:29.803 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.327988, Accuracy: 6.25%
2026-10-04 12:44:30.219 | INFO     | train:train_epoch:47 - Epoch 1, Step 100, Loss: 3.267558, Accuracy: 9.44%
2026-10-04 12:44:30.633 | INFO     | train:train_epoch:47 - Epoch 1, Step 200, Loss: 3.083898, Accuracy: 16.14%
2026-10-04 12:44:31.049 | INFO     | train:train_epoch:47 - Epoch 1, Step 300, Loss: 2.883959, Accuracy: 21.98%
2026-10-04 12:44:31.462 | INFO     | train:test_epoch:90 - Epoch 1, Val Los

In [ ]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-04 13:57:14.160 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet50_seed42_epochs30_initfixed0.78_betalr0.01.log
2026-10-04 13:57:14.232 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 13:57:35.068 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-04 13:57:35.069 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 13:57:35.165 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-04 13:57:35.167 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 13:57:35.167 | INFO     | __main__:main:439 - Total trainable parameters: 57376
2026-10-04 13:57:36.590 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.392796, Accuracy: 3.12%
2026-10-04 13:58:39.133 | INFO     | train:tra

In [4]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-04 21:50:00.797 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet50_seed43_epochs30_initfixed0.78_betalr0.01.log
2026-10-04 21:50:00.869 | INFO     | __main__:main:302 - Running on cuda
2026-10-04 21:50:21.554 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-04 21:50:21.554 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-04 21:50:21.645 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-04 21:50:21.647 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-04 21:50:21.647 | INFO     | __main__:main:439 - Total trainable parameters: 57376
2026-10-04 21:50:23.015 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.292988, Accuracy: 9.38%
2026-10-04 21:51:17.630 | INFO     | train:tra

In [3]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy fixed \
    --init_beta 0.78 \
    --beta_lr 0.01 \
    --epochs 30

2026-10-05 00:09:38.561 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet50_seed44_epochs30_initfixed0.78_betalr0.01.log
2026-10-05 00:09:38.623 | INFO     | __main__:main:302 - Running on cuda
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100%|███████████████████████████████████████| 97.8M/97.8M [00:00<00:00, 216MB/s]
2026-10-05 00:09:56.641 | INFO     | __main__:main:389 - Initialization strategy: fixed
2026-10-05 00:09:56.641 | INFO     | __main__:main:394 - Requested initial stage betas: [0.78, 0.78, 0.78, 0.78]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-05 00:09:56.733 | INFO     | __main__:main:418 - Initial stage betas: [0.7799999713897705, 0.7799999713897705, 0.7799999713897705, 0.7799999713897705]
2026-10-05 00:09:56.734 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-05 00:09:56.734 | INFO  

In [4]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 42 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-05 01:54:22.320 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet50_seed42_epochs30_initrandom_betalr0.01.log
2026-10-05 01:54:22.412 | INFO     | __main__:main:302 - Running on cuda
2026-10-05 01:54:42.494 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-05 01:54:42.495 | INFO     | __main__:main:394 - Requested initial stage betas: [0.924447, 0.827275, 0.948993, 0.902237]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-05 01:54:42.563 | INFO     | __main__:main:418 - Initial stage betas: [0.924447238445282, 0.827274739742279, 0.9489933848381042, 0.9022367000579834]
2026-10-05 01:54:42.564 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-05 01:54:42.565 | INFO     | __main__:main:439 - Total trainable parameters: 57376
2026-10-05 01:54:43.866 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.396827, Accuracy: 3.12%
2026-10-05 01:55:39.927 | INFO    

In [5]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 43 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-05 03:39:10.758 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet50_seed43_epochs30_initrandom_betalr0.01.log
2026-10-05 03:39:10.817 | INFO     | __main__:main:302 - Running on cuda
2026-10-05 03:39:30.782 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-05 03:39:30.782 | INFO     | __main__:main:394 - Requested initial stage betas: [0.889167, 0.712695, 0.705809, 0.943372]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-05 03:39:30.886 | INFO     | __main__:main:418 - Initial stage betas: [0.8891667127609253, 0.7126947641372681, 0.7058085799217224, 0.9433716535568237]
2026-10-05 03:39:30.888 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-05 03:39:30.888 | INFO     | __main__:main:439 - Total trainable parameters: 57376
2026-10-05 03:39:32.200 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.315885, Accuracy: 9.38%
2026-10-05 03:40:28.276 | INFO  

In [6]:
!CUDA_VISIBLE_DEVICES=1 WANDB_MODE=disabled python generalization_stagewise_ct.py \
    --model resnet50 \
    --pretrained_ds imagenet \
    --transfer_ds arabic-characters \
    --seed 44 \
    --transfer_train_bs 32 \
    --transfer_test_bs 128 \
    --init_strategy random \
    --beta_lr 0.01 \
    --epochs 30

2026-10-05 05:23:51.085 | INFO     | __main__:main:286 - Log file: ./logs/generalization_stagewise_ct_imagenet_to_arabic-characters_resnet50_seed44_epochs30_initrandom_betalr0.01.log
2026-10-05 05:23:51.146 | INFO     | __main__:main:302 - Running on cuda
2026-10-05 05:24:08.851 | INFO     | __main__:main:389 - Initialization strategy: random
2026-10-05 05:24:08.852 | INFO     | __main__:main:394 - Requested initial stage betas: [0.735544, 0.774853, 0.817674, 0.981063]
Stage-Wise CT ReLU counts: [13, 16, 24, 12]
2026-10-05 05:24:08.922 | INFO     | __main__:main:418 - Initial stage betas: [0.7355440258979797, 0.7748528122901917, 0.8176735043525696, 0.9810633659362793]
2026-10-05 05:24:08.923 | INFO     | __main__:main:434 - Trainable curvature parameters: 4
2026-10-05 05:24:08.924 | INFO     | __main__:main:439 - Total trainable parameters: 57376
2026-10-05 05:24:10.339 | INFO     | train:train_epoch:47 - Epoch 1, Step 0, Loss: 3.344718, Accuracy: 0.00%
2026-10-05 05:25:06.417 | INFO  